# 📊 Web Scraping — Montos Programación del PMI

**Fuente:** https://ofi5.mef.gob.pe/invierte/pmi/consultapmi?cui={CUI}

Extrae por CUI: **OPMI** y **Montos 2026–2029**

### 📋 Instrucciones
1. Coloca `CUI.xlsx` en la **misma carpeta** que este notebook (columna `CUI`)
2. Edita solo la celda de **Configuración** (PASO 2)
3. Ejecuta en orden con **Shift+Enter**

### ⚙️ PASO 1 — Instalación
> Solo la primera vez.

In [ ]:
!pip install selenium webdriver-manager openpyxl pandas

### ⚙️ PASO 2 — Configuración
> **⚠️ Única celda que debes editar.**

In [ ]:
from pathlib import Path

CARPETA_BASE = Path.cwd()

NOMBRE_ARCHIVO_ENTRADA = "CUI.xlsx"            # <- Cambia si tu archivo tiene otro nombre
NOMBRE_ARCHIVO_SALIDA  = "resultados_PMI.xlsx" # <- Nombre del archivo de resultados

RUTA_ENTRADA = CARPETA_BASE / NOMBRE_ARCHIVO_ENTRADA
RUTA_SALIDA  = CARPETA_BASE / NOMBRE_ARCHIVO_SALIDA

MAX_REINTENTOS = 2
MODO_VISIBLE   = True
GUARDAR_CADA   = 5
TIMEOUT_PAGINA   = 20
TIMEOUT_ELEMENTO = 10

print(f"📁 Carpeta: {CARPETA_BASE}")
print(f"📥 Entrada: {NOMBRE_ARCHIVO_ENTRADA}")
print(f"📤 Salida : {NOMBRE_ARCHIVO_SALIDA}")
print(f"{'✅' if RUTA_ENTRADA.exists() else '❌ NO ENCONTRADO'} {NOMBRE_ARCHIVO_ENTRADA}")

### ⚙️ PASO 3 — Importar librerías

In [ ]:
import pandas as pd
import time
from selenium import webdriver
from selenium.webdriver import Chrome
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait as Wait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException
from webdriver_manager.chrome import ChromeDriverManager
print("✅ Librerías listas")

### ⚙️ PASO 4 — Funciones de checkpoint y navegador

In [ ]:
def cargar_progreso():
    if RUTA_SALIDA.exists():
        try:
            df = pd.read_excel(RUTA_SALIDA)
            print(f"📥 Progreso: {len(df)} CUIs ya procesados")
            return df.to_dict('records')
        except Exception as e:
            print(f"⚠️ Error leyendo progreso: {e}")
    return []

def obtener_pendientes(completa, procesados):
    if not procesados:
        print(f"📋 Todos pendientes: {len(completa)}")
        return completa, {}
    procesados_dict = {str(r['CUI']): r for r in procesados}
    nuevos = [cui for cui in completa if str(cui) not in procesados_dict]
    if nuevos:
        print(f"📋 CUIs pendientes: {len(nuevos)}")
    return nuevos, procesados_dict

def guardar(resultados):
    try:
        pd.DataFrame(resultados).to_excel(RUTA_SALIDA, index=False)
    except Exception as e:
        print(f"⚠️ Error guardando: {e}")

def crear_driver():
    service = Service(ChromeDriverManager().install())
    options = webdriver.ChromeOptions()
    if not MODO_VISIBLE:
        options.add_argument("--headless=new")
    else:
        options.add_argument("--start-maximized")
    for arg in ["--disable-gpu","--no-sandbox","--disable-dev-shm-usage",
                "--disable-extensions","--disable-blink-features=AutomationControlled",
                "--disable-logging","--log-level=3","--silent"]:
        options.add_argument(arg)
    prefs = {"profile.managed_default_content_settings.images": 2,
             "profile.default_content_setting_values.notifications": 2}
    options.add_experimental_option("prefs", prefs)
    options.add_experimental_option('excludeSwitches', ['enable-logging'])
    options.page_load_strategy = 'normal'
    driver = Chrome(service=service, options=options)
    driver.set_page_load_timeout(TIMEOUT_PAGINA)
    driver.set_script_timeout(15)
    return driver

print("✅ Funciones listas")

### ⚙️ PASO 5 — Funciones de extracción (PMI)

In [ ]:
def cui_existe_en_pmi(driver):
    try:
        page_text = driver.find_element(By.TAG_NAME, "body").text.lower()
        for error in ["no se encontró","no existe","sin registros","cui no válido"]:
            if error in page_text:
                return False
        if len(page_text.strip()) < 100:
            return False
        tabla = driver.find_element(By.ID, "tblResultado")
        tbody = tabla.find_element(By.TAG_NAME, "tbody")
        return len(tbody.find_elements(By.TAG_NAME, "tr")) > 0
    except:
        return False

def extraer_datos_pmi(driver):
    """
    Col 4 (índ.3): OPMI | Cols 13-16 (índ.12-15): Montos 2026-2029
    """
    try:
        time.sleep(2)
        celdas = (driver.find_element(By.ID, "tblResultado")
                       .find_element(By.TAG_NAME, "tbody")
                       .find_element(By.TAG_NAME, "tr")
                       .find_elements(By.TAG_NAME, "td"))
        if len(celdas) < 16:
            print(f"⚠️ Solo {len(celdas)} cols", end=" ")
            return None
        def c(i): return celdas[i].text.strip() or 'NO DISPONIBLE'
        datos = {'opmi': c(3), 'monto_2026': c(12), 'monto_2027': c(13),
                 'monto_2028': c(14), 'monto_2029': c(15)}
        if datos['opmi'] == 'NO DISPONIBLE':
            return None
        return datos
    except Exception as e:
        print(f"⚠️ {str(e)[:30]}", end=" ")
        return None

def procesar_cui(driver, cui, intento=1):
    try:
        driver.get(f"https://ofi5.mef.gob.pe/invierte/pmi/consultapmi?cui={cui}")
        Wait(driver, TIMEOUT_ELEMENTO).until(EC.presence_of_element_located((By.TAG_NAME, "body")))
        time.sleep(1.5)
        if not cui_existe_en_pmi(driver):
            return None, "CUI_NO_EXISTE"
        datos = extraer_datos_pmi(driver)
        if not datos:
            if intento < MAX_REINTENTOS:
                print(f"🔄{intento+1}...", end=" ")
                time.sleep(1)
                return procesar_cui(driver, cui, intento + 1)
            return None, "ERROR_EXTRACCION"
        return datos, "EXITO"
    except:
        if intento < MAX_REINTENTOS:
            print(f"🔄{intento+1}...", end=" ")
            time.sleep(1)
            return procesar_cui(driver, cui, intento + 1)
        return None, "ERROR_CONEXION"

def agregar_resultado(resultados, cui, datos, estado):
    if estado == "EXITO":
        reg = {"CUI": cui, "OPMI": datos['opmi'],
               "Monto año 2026": datos['monto_2026'],
               "Monto año 2027": datos['monto_2027'],
               "Monto año 2028": datos['monto_2028'],
               "Monto año 2029": datos['monto_2029']}
    else:
        reg = {"CUI": cui, "OPMI": "NO DISPONIBLE",
               "Monto año 2026": "NO DISPONIBLE",
               "Monto año 2027": "NO DISPONIBLE",
               "Monto año 2028": "NO DISPONIBLE",
               "Monto año 2029": "NO DISPONIBLE"}
    for i, r in enumerate(resultados):
        if str(r['CUI']) == str(cui):
            resultados[i] = reg
            return
    resultados.append(reg)

print("✅ Funciones de extracción PMI listas")

### ▶️ PASO 6 — Cargar CUIs e inicializar

In [ ]:
print("📂 Cargando CUIs...")
df_cui = pd.read_excel(RUTA_ENTRADA)
lista_completa = df_cui['CUI'].astype(str).tolist()
print(f"📊 Total: {len(lista_completa)}")

resultados_previos = cargar_progreso()
lista_cuis, procesados_dict = obtener_pendientes(lista_completa, resultados_previos)

if not lista_cuis:
    print("🎉 ¡Todos los CUIs ya fueron procesados!")
else:
    resultados = resultados_previos.copy()
    driver = crear_driver()
    print(f"⚡ Listo para procesar {len(lista_cuis)} CUIs")

### ▶️ PASO 7 — Ejecutar el scraping
> ⚠️ No cierres Chrome. Interrumpe con ⏹️; re-ejecuta PASOS 6 y 7 para continuar.

In [ ]:
tiempo_inicio    = time.time()
exitosos         = 0
fallos_no_existe = 0
fallos_otros     = 0

print(f"{'='*60}\nInicio: {time.strftime('%H:%M:%S')}\n{'='*60}\n")

try:
    for idx, cui in enumerate(lista_cuis, 1):
        t0 = time.time()
        print(f"🆕 [{idx}/{len(lista_cuis)}] {cui}:", end=" ")
        try:
            datos, estado = procesar_cui(driver, cui)
            agregar_resultado(resultados, cui, datos, estado)
            if estado == "EXITO":
                exitosos += 1
                print(f"✅ {datos['opmi'][:30]} ({time.time()-t0:.1f}s)")
            elif estado == "CUI_NO_EXISTE":
                fallos_no_existe += 1
                print(f"⚪ No en PMI ({time.time()-t0:.1f}s)")
            else:
                fallos_otros += 1
                print(f"❌ {estado} ({time.time()-t0:.1f}s)")
        except Exception as e:
            agregar_resultado(resultados, cui, None, "ERROR")
            fallos_otros += 1
            print(f"❌ {str(e)[:30]} ({time.time()-t0:.1f}s)")

        if (idx % GUARDAR_CADA == 0) or (idx == len(lista_cuis)):
            guardar(resultados)
            if idx % GUARDAR_CADA == 0:
                print(f"   💾 Guardado")

        if idx % 10 == 0:
            t_el = time.time() - tiempo_inicio
            eta  = (len(lista_cuis) - idx) * (t_el / idx)
            print(f"   📊 ✅{exitosos} ⚪{fallos_no_existe} ❌{fallos_otros} | {t_el:.0f}s | ETA:{eta:.0f}s\n")

except KeyboardInterrupt:
    print("\n⚠️ INTERRUMPIDO")
    guardar(resultados)
    driver.quit()
    print("💾 Guardado. Re-ejecuta PASOS 6 y 7 para continuar.")
else:
    driver.quit()
    guardar(resultados)
    t_total = time.time() - tiempo_inicio
    print(f"\n{'='*60}\n🏁 COMPLETADO\n{'='*60}")
    print(f"📊 Procesados: {len(lista_cuis)} | ✅ {exitosos} | ⚪ {fallos_no_existe} | ❌ {fallos_otros}")
    print(f"⏱️  {t_total:.1f}s ({t_total/60:.1f} min) | 💾 {RUTA_SALIDA}")

### 📊 PASO 8 — Estadísticas (opcional)

In [ ]:
if RUTA_SALIDA.exists():
    df_final = pd.read_excel(RUTA_SALIDA)
    total    = len(df_final)
    ok       = len(df_final[df_final['OPMI'] != 'NO DISPONIBLE'])
    print(f"Total: {total} | ✅ Con datos: {ok} ({ok/total*100:.1f}%) | ❌ Sin datos: {total-ok}")
    if ok > 0:
        for anio in ['2026','2027','2028','2029']:
            col = f'Monto año {anio}'
            n = len(df_final[(df_final[col]!='NO DISPONIBLE')&(df_final[col]!='0.0')])
            print(f"  {anio}: {n} CUIs con monto ({n/ok*100:.1f}%)")
    df_final.head(5)
else:
    print("⚠️ Ejecuta el scraping primero.")